In [1]:
%pip install gradio insightface onnxruntime opencv-python-headless pillow "numpy<2" open_clip_torch torch --quiet

Note: you may need to restart the kernel to use updated packages.


##### Load both models

In [2]:
import pickle
from pathlib import Path

import numpy as np
import torch
import open_clip
import gradio as gr
from PIL import Image
from insightface.app import FaceAnalysis


# ArcFace
print("Loading ArcFace (buffalo_s)...")
app = FaceAnalysis(name="buffalo_s", providers=["CPUExecutionProvider"])
app.prepare(ctx_id=0, det_size=(320, 320))

# CLIP
print("Loading CLIP (ViT-B/32)...")
clip_device = "cpu"
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="laion2b_s34b_b79k"
)
clip_model.eval().to(clip_device)

print("Both models ready.")

/Users/brandley/.pyenv/versions/3.11.3/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading ArcFace (buffalo_s)...
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/brandley/.insightface/models/buffalo_s/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/brandley/.insightface/models/buffalo_s/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/brandley/.insightface/models/buffalo_s/det_500m.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/brandley/.insightface/models/buffalo_s/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/brandley/.insightface/models/buffalo_s/w600k_mbf.onnx r

Both models ready.


In [3]:
# Folder with player photos (used to fetch representative images at display time)
IMAGES_DIR    = Path("images")

# Files produced by model_builder.ipynb
ARC_FILE      = Path("player_embeddings.npz")
ARC_PER_IMAGE = Path("per_image_embeddings.npz")
CLIP_FILE     = Path("clip_player_embeddings.npz")
KEPT_MAP_FILE = Path("kept_map.pkl")

# Matching parameters
TOP_K         = 3
ARC_WEIGHT    = 0.3
CLIP_WEIGHT   = 0.7

In [4]:
# ArcFace per-player means
arc_data = np.load(ARC_FILE)
clean = {k: arc_data[k] for k in arc_data.files}
print(f"Loaded {len(clean)} ArcFace fingerprints from {ARC_FILE}")

# CLIP per-player means
clip_data = np.load(CLIP_FILE)
clip_clean = {k: clip_data[k] for k in clip_data.files}
print(f"Loaded {len(clip_clean)} CLIP fingerprints from {CLIP_FILE}")

# kept_map — which filenames passed the outlier filter for each player
with open(KEPT_MAP_FILE, "rb") as f:
    kept_map = pickle.load(f)
print(f"Loaded kept_map for {len(kept_map)} players from {KEPT_MAP_FILE}")

# per_image — needed by best_kept_image() to pick the most representative photo per player
# The npz keys are "{player_key}__{filename}", so split each one to reconstruct the dict
per_image = {}
per_image_data = np.load(ARC_PER_IMAGE)
for combined_key in per_image_data.files:
    player_key, filename = combined_key.rsplit("__", 1)
    per_image.setdefault(player_key, []).append((filename, per_image_data[combined_key]))
print(f"Loaded per-image vectors for {len(per_image)} players from {ARC_PER_IMAGE}")

# Pre-compute the matrices used for fast matching at request time
COMMON_KEYS = sorted(set(clean.keys()) & set(clip_clean.keys()))
ARC_MATRIX  = np.stack([clean[k]      for k in COMMON_KEYS])
CLIP_MATRIX = np.stack([clip_clean[k] for k in COMMON_KEYS])
print(f"\nReady to match against {len(COMMON_KEYS)} players.")

Loaded 1205 ArcFace fingerprints from player_embeddings.npz
Loaded 1205 CLIP fingerprints from clip_player_embeddings.npz
Loaded kept_map for 1205 players from kept_map.pkl
Loaded per-image vectors for 1205 players from per_image_embeddings.npz

Ready to match against 1205 players.


In [5]:
def best_kept_image(player_key):
    """Return the filename of the most representative kept image for a player."""
    items = per_image.get(player_key, [])
    kept_set = set(kept_map.get(player_key, []))
    mean = clean[player_key]
    candidates = [(f, float(e @ mean)) for f, e in items if f in kept_set]
    if not candidates:
        return None
    return max(candidates, key=lambda x: x[1])[0]


@torch.no_grad()
def clip_embed_path(img_path):
    """Return the L2-normalized 512-d CLIP image embedding."""
    with Image.open(img_path) as pil:
        pil = pil.convert("RGB")
        tensor = clip_preprocess(pil).unsqueeze(0).to(clip_device)
    feat = clip_model.encode_image(tensor)
    feat = feat / feat.norm(dim=-1, keepdim=True)
    return feat.squeeze(0).cpu().numpy()


def find_lookalike_combined(selfie_path, top_k=TOP_K):
    """Return top-k matches as (player_key, combined_score, arc_score, clip_score)."""
    # ArcFace embedding
    img = np.array(Image.open(selfie_path).convert("RGB"))
    faces = app.get(img)
    if not faces:
        return None
    faces.sort(key=lambda f: (f.bbox[2]-f.bbox[0]) * (f.bbox[3]-f.bbox[1]),
               reverse=True)
    q_arc = faces[0].normed_embedding

    # CLIP embedding
    q_clip = clip_embed_path(selfie_path)

    # Score against every player
    arc_scores  = ARC_MATRIX  @ q_arc
    clip_scores = CLIP_MATRIX @ q_clip

    # Normalize both score sets to [0, 1] so the weights work as expected
    arc_n  = (arc_scores  - arc_scores.min())  / (arc_scores.ptp()  + 1e-9)
    clip_n = (clip_scores - clip_scores.min()) / (clip_scores.ptp() + 1e-9)
    combined = ARC_WEIGHT * arc_n + CLIP_WEIGHT * clip_n

    top = np.argsort(-combined)[:top_k]
    return [(COMMON_KEYS[i],
             float(combined[i]),
             float(arc_scores[i]),
             float(clip_scores[i])) for i in top]


def gradio_lookalike(selfie_path):
    """Gradio handler — returns three (image, caption) pairs and a status message."""
    blank = [None, "", None, "", None, ""]
    if selfie_path is None:
        return [*blank, "Please upload a selfie first."]

    results = find_lookalike_combined(selfie_path)
    if results is None:
        return [*blank, "No face detected — try a clearer, forward-facing photo."]

    images   = [None, None, None]
    captions = ["",   "",   ""]
    for i, (player_key, combined, arc_s, clip_s) in enumerate(results):
        team, name = player_key.split("__", 1)
        rep_file = best_kept_image(player_key)
        if not rep_file:
            continue
        rep_path = IMAGES_DIR / team / name / rep_file
        if not rep_path.exists():
            continue
        pct = int(round(combined * 100))
        images[i] = str(rep_path)
        captions[i] = (
            f"### {name.replace('_', ' ')}\n"
            f"**{team}**  \n"
            f"{pct}% match"
        )
    return [images[0], captions[0],
            images[1], captions[1],
            images[2], captions[2],
            ""]

In [6]:
import shutil
from pathlib import Path
import numpy as np

APP_DIR = Path("hf_space")
REP_DIR = APP_DIR / "representatives"
APP_DIR.mkdir(exist_ok=True)
REP_DIR.mkdir(exist_ok=True)

# Save embeddings into the deployment folder
np.savez(APP_DIR / "player_embeddings.npz",
         **{p: v.astype(np.float32) for p, v in clean.items()})
np.savez(APP_DIR / "clip_player_embeddings.npz",
         **{p: v.astype(np.float32) for p, v in clip_clean.items()})

# Precompute and copy each player's most representative photo (flat naming)
def _best(player_key):
    items = per_image.get(player_key, [])
    kept_set = set(kept_map.get(player_key, []))
    mean = clean[player_key]
    candidates = [(f, float(e @ mean)) for f, e in items if f in kept_set]
    return max(candidates, key=lambda x: x[1])[0] if candidates else None

reps = 0
common = set(clean.keys()) & set(clip_clean.keys())
for player_key in common:
    rep = _best(player_key)
    if not rep:
        continue
    team, name = player_key.split("__", 1)
    src = IMAGES_DIR / team / name / rep
    dst = REP_DIR / f"{player_key}.jpg"
    if src.exists():
        shutil.copy(src, dst)
        reps += 1

bundle_size = sum(p.stat().st_size for p in APP_DIR.rglob("*")) / 1_048_576
print(f"Packaged {reps} representatives into {REP_DIR}/")
print(f"Bundle size: {bundle_size:.1f} MB")
print(f"\nFolder contents:")
for p in sorted(APP_DIR.iterdir()):
    print(f"  {p.name}")

Packaged 1205 representatives into hf_space/representatives/
Bundle size: 748.8 MB

Folder contents:
  .venv
  app.py
  clip_player_embeddings.npz
  player_embeddings.npz
  representatives
  requirements.txt


In [7]:
arc_pct  = int(round(ARC_WEIGHT  * 100))
clip_pct = int(round(CLIP_WEIGHT * 100))

# Cleanly close any previous Gradio server before launching a new one
try:
    demo.close()
except Exception:
    pass

with gr.Blocks(
    title=f"Footballer Lookalike — ArcFace {arc_pct}% / CLIP {clip_pct}%",
    theme=gr.themes.Soft()
) as demo:

    gr.Markdown(
        f"# Which footballer do you look like? ⚽️👤  "
        f"<span style='font-size:0.55em; opacity:0.65; font-weight:normal;'>"
        f"ArcFace {arc_pct}% &nbsp;·&nbsp; CLIP {clip_pct}%</span>"
    )
    gr.Markdown(
        "Upload a clear, forward-facing selfie or use your camera. "
        "We combine **facial structure** (ArcFace) with **appearance similarity** "
        "(CLIP) to find your closest match. Your picture is not saved or shared."
    )

    with gr.Row():
        with gr.Column(scale=1):
            selfie_in  = gr.Image(type="filepath", label="Your selfie",
                                  sources=["upload", "webcam"])
            submit_btn = gr.Button("Find my lookalikes", variant="primary")

        with gr.Column(scale=3):
            with gr.Row():
                with gr.Column():
                    img1 = gr.Image(show_label=False, height=240, interactive=False)
                    cap1 = gr.Markdown("")
                with gr.Column():
                    img2 = gr.Image(show_label=False, height=240, interactive=False)
                    cap2 = gr.Markdown("")
                with gr.Column():
                    img3 = gr.Image(show_label=False, height=240, interactive=False)
                    cap3 = gr.Markdown("")
            status_out = gr.Markdown("")

    gr.Markdown(
        f"<div style='text-align:center; font-size:0.85em; opacity:0.6; "
        f"margin-top:1em;'>Model weights: ArcFace {arc_pct}% &nbsp;·&nbsp; "
        f"CLIP {clip_pct}%</div>"
    )

    submit_btn.click(
        gradio_lookalike,
        inputs=[selfie_in],
        outputs=[img1, cap1, img2, cap2, img3, cap3, status_out]
    )

demo.launch()

/var/folders/0g/v46xc7wn14g9tc8__fwrprm00000gn/T/ipykernel_35838/4143936804.py:10: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [7]:
pip install gradio

Note: you may need to restart the kernel to use updated packages.


In [9]:
source .venv/bin/activate

NameError: name 'source' is not defined